# Bài 6: Vector riêng và Giá trị riêng của Ma trận Hiệp phương sai Mô-men

Ở Bài 5, chúng ta dùng mô-men trung tâm để tìm hướng của một đốm và, với thêm chút biến đổi đại số, độ dài các bán trục của nó. Bài này làm cho mối liên hệ đó trở nên tường minh và tổng quát: các mô-men trung tâm của *bất kỳ* đốm nào (không chỉ elip) xác định một **ma trận hiệp phương sai (covariance matrix)** $2\times2$, và các vector riêng/giá trị riêng của ma trận đó trực tiếp cho ra hướng và kích thước của *elip tương đương (equivalent ellipse)* &mdash; elip có cùng diện tích và cùng độ trải mô-men bậc hai với đốm.

In [ ]:
# Download the sample images used in this lesson, if not already present
import os, urllib.request

for _f in ['glasses_outline.png']:
    _path = f"../img/{_f}"
    if not os.path.exists(_path):
        os.makedirs("../img", exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/sbirchfield/sbirchfield.github.io/main/cvintro/img/{_f}", _path)

## Ôn lại: vector riêng và giá trị riêng là gì?

Với một ma trận vuông $A$, vector khác không $v$ là một **vector riêng (eigenvector)** ứng với **giá trị riêng (eigenvalue)** $\lambda$ nếu

$$Av = \lambda v$$

Nói cách khác: áp dụng $A$ lên $v$ không làm $v$ xoay lệch khỏi đường thẳng của chính nó &mdash; nó chỉ *co giãn* $v$ theo hệ số $\lambda$. Hầu hết các vector đều vừa bị xoay vừa bị co giãn bởi $A$; vector riêng là những hướng đặc biệt chỉ bị co giãn.

Với một ma trận **đối xứng** $2\times2$ (như các ma trận hiệp phương sai mà ta sẽ xây dựng bên dưới), một điều còn tốt hơn được đảm bảo: luôn có hai vector riêng, chúng vuông góc với nhau, và các giá trị riêng là số thực. Về mặt hình học, $A$ biến một đường tròn gồm các vector đơn vị thành một elip có các trục hướng theo vector riêng, với độ dài mỗi bán trục bằng giá trị riêng tương ứng. Hình ảnh đó &mdash; vector riêng là *hướng* của trục, giá trị riêng là *độ dài* của trục &mdash; chính là điều ta sẽ khai thác để phân tích hình dạng bên dưới.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

A = np.array([[3.0, 1.0],
              [1.0, 1.5]])

eigvals, eigvecs = np.linalg.eigh(A)  # ascending order
print('eigenvalues :', eigvals)
print('eigenvectors (columns):\n', eigvecs)

theta = np.linspace(0, 2 * np.pi, 200)
circle = np.stack([np.cos(theta), np.sin(theta)])  # unit circle, as column vectors
ellipse = A @ circle  # apply A to every point on the circle

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot(circle[0], circle[1], '--', color='gray', label='unit circle')
ax.plot(ellipse[0], ellipse[1], color='#3498db', label='A @ circle')
for val, vec in zip(eigvals, eigvecs.T):
    ax.plot([0, val * vec[0]], [0, val * vec[1]], color='red', linewidth=2)
ax.scatter(0, 0, color='black', zorder=5)
ax.set_aspect('equal')
ax.legend(loc='upper left', fontsize=8)
ax.set_title('A circle stretched by A; red lines = eigenvectors x eigenvalues')
plt.show()

Trong ảnh động này, một đường tròn các vector được biến đổi dần qua A, từ ma trận đơn vị (lúc đầu) đến chính A (lúc cuối). Hãy quan sát cách mọi vector thông thường (màu xám) xoay lệch khỏi hướng ban đầu khi đường tròn biến dạng thành elip, trong khi hai vector riêng (đỏ, lam) luôn nằm trên cùng một đường thẳng — chúng chỉ giãn ra, đúng bằng giá trị riêng của mình.

<img src="../animations/eigen_demo.gif" alt="Ảnh động minh họa một ma trận biến đường tròn thành elip, cho thấy vector riêng không bị xoay" style="display: block; margin: 0 auto; max-width: 650px;">

### Từ một vector riêng đến cả hai cùng lúc: chéo hóa

Đẳng thức $Av = \lambda v$ ở trên đúng với một vector riêng. Hãy xếp *cả hai* vector riêng thành các cột của ma trận $P = \begin{bmatrix}v_1 & v_2\end{bmatrix}$, và cả hai giá trị riêng trên đường chéo của $\Lambda = \begin{bmatrix}\lambda_1 & 0\\0 & \lambda_2\end{bmatrix}$. Khi đó $Av_1=\lambda_1 v_1$ và $Av_2=\lambda_2 v_2$, đặt cạnh nhau, trở thành một phương trình ma trận duy nhất:

$$AP = P\Lambda$$

Vì $A$ đối xứng, các vector riêng của nó không chỉ vuông góc mà còn *trực chuẩn (orthonormal)*, khiến $P$ là một ma trận **trực giao (orthogonal)** (nghịch đảo của nó chỉ là chuyển vị: $P^{-1} = P^\top$). Nhờ vậy ta giải được chính $A$:

$$A = P\Lambda P^\top$$

Đây là phép **chéo hóa (diagonalization)** của $A$, và về hình học nó gồm ba bước: $P^\top$ *xoay* hệ tọa độ vào khung vector riêng, $\Lambda$ *co giãn* độc lập dọc theo các hướng đó (lúc này đã thẳng hàng với trục), và $P$ *xoay ngược lại*. Nhìn trong khung đã xoay đó, $A$ thực sự là ma trận chéo &mdash; phần tử ngoài đường chéo, thứ trộn lẫn $x$ và $y$, bằng đúng 0. Khi $A$ là ma trận hiệp phương sai, phần tử ngoài đường chéo đó chính là $\mu_{11}$: chéo hóa ma trận hiệp phương sai và tìm các trục tự nhiên của elip là cùng một phép toán &mdash; cả hai đều có nghĩa là xoay vào hệ tọa độ duy nhất mà ở đó độ trải của đốm theo $x$ và theo $y$ hoàn toàn không còn trộn lẫn.

In [ ]:
P = eigvecs
Lam = np.diag(eigvals)

print('A @ P:\n', np.round(A @ P, 4))
print('P @ Lambda:\n', np.round(P @ Lam, 4))
print('max |A@P - P@Lambda| (should be ~0):', np.abs(A @ P - P @ Lam).max())
print()

reconstructed = P @ Lam @ P.T
print('P @ Lambda @ P.T (reconstructed A):\n', np.round(reconstructed, 4))
print('original A:\n', A)
print('max |reconstructed - A|:', np.abs(reconstructed - A).max())
print()

diagonalized = P.T @ A @ P  # A, read in the eigenvector frame
print('P.T @ A @ P (A in its own eigenbasis -- should be diagonal):\n', np.round(diagonalized, 4))

Trong ảnh động này, cùng ma trận A được áp dụng thành ba bước riêng biệt thay vì một: đầu tiên Pᵀ xoay các vector riêng về trục x/y, sau đó Λ giãn dọc theo các trục đó theo giá trị riêng, rồi P xoay mọi thứ trở lại. Ma trận đang được áp dụng được tô màu cam. Kết quả trùng khớp với đúng elip thu được khi áp dụng A trực tiếp ở trên.

<img src="../animations/diagonalization_demo.gif" alt="Ảnh động minh họa phép chéo hóa A = P Lambda P chuyển vị, áp dụng thành ba bước tuần tự" style="display: block; margin: 0 auto; max-width: 750px;">

## Từ mô-men trung tâm đến ma trận hiệp phương sai

Hãy coi các điểm ảnh của một đốm là các mẫu lấy từ một phân phối 2D. Ma trận hiệp phương sai của nó, biểu diễn theo các mô-men trung tâm $\mu_{ij}$ và diện tích $\mu_{00}=m_{00}$, là

$$A = \frac{1}{m_{00}}\begin{bmatrix}\mu_{20} & \mu_{11} \\ \mu_{11} & \mu_{02}\end{bmatrix}$$

Đây chính là công thức dùng cho ma trận hiệp phương sai của một tập điểm phân tán $(x,y)$, chỉ khác là được đánh trọng số theo việc điểm ảnh có thuộc đốm hay không thay vì theo chỉ số mẫu. Số hạng ngoài đường chéo $\mu_{11}$ chính là phần tử còn sót lại do sự trộn lẫn $x$ và $y$, nên đây đúng là ma trận ta muốn **chéo hóa**: các vector riêng của nó sẽ hướng dọc theo trục chính và trục phụ của đốm &mdash; các hướng có độ trải lớn nhất và nhỏ nhất &mdash; và các giá trị riêng sẽ đo độ trải dọc theo mỗi hướng. Cũng phép toán này, áp dụng cho ma trận hiệp phương sai của dữ liệu tổng quát (không nhất thiết là điểm ảnh 2D), được gọi là **phân tích thành phần chính (principal component analysis — PCA)**: các vector riêng xếp theo giá trị riêng là các *thành phần chính (principal components)*, các hướng có phương sai lớn nhất trong dữ liệu, và thường được dùng để giảm một tập dữ liệu nhiều chiều xuống chỉ còn vài hướng giàu thông tin nhất.

In [ ]:
def covariance_from_moments(binary):
    m = cv2.moments(binary, binaryImage=True)
    cx, cy = m['m10'] / m['m00'], m['m01'] / m['m00']
    cov = np.array([[m['mu20'], m['mu11']],
                     [m['mu11'], m['mu02']]]) / m['m00']
    return cov, (cx, cy)

## Phân rã riêng cho ra hướng và kích thước

`np.linalg.eigh` trả về các giá trị riêng theo thứ tự tăng dần cùng với các vector riêng (dưới dạng cột). Với một elip đặc có các bán trục $a \ge b$, các giá trị riêng tính ra là $\lambda_{\max} = a^2/4$ và $\lambda_{\min}=b^2/4$, do đó

$$a = 2\sqrt{\lambda_{\max}}, \qquad b = 2\sqrt{\lambda_{\min}}$$

và các vector riêng tương ứng hướng dọc theo trục chính và trục phụ. Lưu ý rằng góc của trục chính trùng với góc đã tính ở Bài 5.

*Hãy để ý căn bậc hai ở đây, thứ không có trong phần ôn tập ở trên: ở đó, các giá trị riêng của ma trận tùy ý *A* chính là các hệ số giãn áp dụng lên đường tròn đơn vị. Còn ở đây, các giá trị riêng của ma trận hiệp phương sai là phương sai (được xây dựng từ bình phương độ lệch tọa độ), nên để khôi phục độ dài trục thực sự ta phải hoàn tác phép bình phương đó, vì thế mới có căn bậc hai.*

In [ ]:
def principal_axes(binary):
    cov, center = covariance_from_moments(binary)
    eigvals, eigvecs = np.linalg.eigh(cov)  # ascending order
    semi_axes = 2 * np.sqrt(np.clip(eigvals, 0, None))
    # reorder so index 0 is major (largest), index 1 is minor
    order = [1, 0]
    # eigvecs[:, order] has the two eigenvectors as its COLUMNS; transpose so that
    # unpacking it below (which iterates over ROWS) hands back the two actual eigenvectors,
    # not a scrambled mix of their x- and y-components
    return center, semi_axes[order], eigvecs[:, order].T


def draw_axes(ax, binary, color='red'):
    center, (a, b), (v_major, v_minor) = principal_axes(binary)
    cx, cy = center
    ax.imshow(binary, cmap='gray')
    for length, vec, lw in [(a, v_major, 2.5), (b, v_minor, 1.5)]:
        dx, dy = length * vec
        ax.plot([cx - dx, cx + dx], [cy - dy, cy + dy], c=color, linewidth=lw)
    ax.scatter(cx, cy, c=color, marker='x', s=60)
    ax.axis('off')
    return center, (a, b)

### Kiểm tra: một elip đã biết

Giống như Bài 5, ta vẽ một elip với các tham số đã biết và xác nhận rằng ước lượng dựa trên phân rã riêng khôi phục được chúng &mdash; nhưng lần này ta vẽ *cả hai* trục, không chỉ trục chính.

In [ ]:
binary = np.zeros((200, 200), dtype=np.uint8)
cv2.ellipse(binary, (100, 100), (70, 25), 30, 0, 360, 255, -1)

fig, ax = plt.subplots(figsize=(4, 4))
center, (a, b) = draw_axes(ax, binary)
ax.set_title('Major (thick) and minor (thin) axes from eigenvectors')
plt.show()

print(f'recovered semi-axes: a={a:.1f}, b={b:.1f}  (drawn with 70, 25)')

## Phương pháp cũng hoạt động với hình dạng bất kỳ

Sức mạnh thực sự của cách tiếp cận này là nó hoàn toàn không đòi hỏi đốm phải là elip. Mọi hình nhị phân đều có *một* elip tương đương nào đó &mdash; elip khớp với diện tích, trọng tâm và độ trải mô-men bậc hai của nó. Điều này cho ra một bản tóm tắt gọn gàng gồm 5 con số (tâm, độ dài hai bán trục, hướng) cho một đốm có hình dạng tùy ý.

In [ ]:
im_glasses = cv2.imread('../img/glasses_outline.png', cv2.IMREAD_GRAYSCALE)

shapes = {'Glasses': im_glasses}

fig, ax = plt.subplots(figsize=(5, 4))
center, (a, b) = draw_axes(ax, im_glasses)
ax.set_title(f'Glasses with axes\na={a:.0f}, b={b:.0f}', fontsize=10)
plt.show()

<p class="image-attribution">Nguồn ảnh: Stan Birchfield</p>

Hãy để ý rằng elip tương đương không cố gắng bám theo đường biên của hình &mdash; nó tóm tắt *phân bố khối lượng* quanh trọng tâm, nên nó cắt ngang qua cầu nối hẹp giữa hai mắt kính thay vì đi theo nó, và đường viền của nó nằm ra ngoài hai mắt kính ở gần phía trên và phía dưới, nơi chiếc kính thon lại.

## Độ lệch tâm: một hình thuôn dài đến mức nào?

Tỉ số giữa các giá trị riêng (hoặc các bán trục) cho ra một con số duy nhất mô tả độ thuôn dài, không phụ thuộc vào hướng và kích thước tổng thể:

$$\text{eccentricity} = \sqrt{1 - \frac{\lambda_{\min}}{\lambda_{\max}}}$$

Giá trị này nằm trong khoảng từ 0 (hình tròn, hai trục bằng nhau) đến gần 1 (một hình rất mảnh, thuôn dài).

In [ ]:
print(f'{"shape":>10} {"a":>6} {"b":>6} {"eccentricity":>13}')
for name, img in shapes.items():
    _, (a, b), _ = principal_axes(img)
    ecc = np.sqrt(1 - (b / a) ** 2)
    print(f'{name:>10} {a:6.1f} {b:6.1f} {ecc:13.3f}')

### Bài tập

1. Vẽ một hình tròn hoàn hảo và xác nhận độ lệch tâm (eccentricity) của nó (gần) bằng 0.
2. Vẽ chồng đường viền elip khớp *thực sự* (không chỉ các đường trục) bằng `cv2.ellipse` với tâm, `(2a, 2b)` là độ dài đầy đủ của các trục, và góc hướng lấy từ các vector riêng. So sánh nó với `cv2.fitEllipse` áp dụng lên đường viền của hình &mdash; chúng có khớp nhau không?
3. Chạy `principal_axes` trên đốm đặc của `glasses` (sau khi tô loang). `a`, `b` và độ lệch tâm thay đổi bao nhiêu? Điều đó có khớp với trực giác của bạn rằng mô-men phụ thuộc vào việc khối lượng nằm *ở đâu*, chứ không chỉ vào hình bóng tổng thể không?